# 🎬 Media Downloader (Google Colab 雲端直存與遠端工作節點)

**雙模式支援：**
1. **🤖 模式一 (推薦)：Streamlit 本機聯動背景工作節點**：Colab 掛載後作為雲端 Worker 待命，您只需在 Mac 本地 Streamlit 介面貼網址按下載，Colab 自動在雲端抓取、封裝、直存 Google Drive，並將即時進度同步回您的 Mac 網頁！
2. **🖐️ 模式二：Colab 網頁獨立執行**：直接在下方表單填寫網址與儲存資料夾，點擊三角形按鈕直接下載。

## 步驟 1：安裝環境與掛載 Google Drive
執行下方儲存格安裝必要套件，並點擊授權掛載 Google Drive（兩模式皆需執行一次）。

In [ ]:
#@title ⚙️ 1. 環境安裝與掛載 Google Drive
# 安裝解析與高速下載依賴 (Colab 預設已內建 FFmpeg)
!pip install -q yt-dlp curl_cffi requests beautifulsoup4

from google.colab import drive
import os, sys

# 掛載 Google Drive
drive_mount_path = '/content/drive'
if not os.path.exists(drive_mount_path):
    drive.mount(drive_mount_path)
    print("✅ Google Drive 掛載成功！")
else:
    print("ℹ️ Google Drive 已在掛載狀態。")

# 載入 core_downloader 核心引擎
import urllib.request
raw_url = 'https://raw.githubusercontent.com/hangout1997/Media-Download/main/core_downloader.py'
print("正在由 GitHub 載入最新 core_downloader.py...")
urllib.request.urlretrieve(raw_url, '/content/core_downloader.py')
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
import core_downloader as cd
print("✅ 成功載入 core_downloader 核心引擎！")



---
## 🌟 模式一：啟動 Streamlit 聯動工作節點 (推薦)
執行下方儲存格後，Colab 會進入**待命監聽模式**。您只需回到 Mac 本機 Streamlit 網頁輸入網址，Colab 就會自動接單下載並回傳進度！

In [ ]:
#@title 🤖 啟動 Worker 待命監聽 (與 Mac 本地 Streamlit 聯動)
import os, time, json, glob, sys, shutil, re, traceback

# 1. 自動偵測 Google Drive 掛載根目錄
def get_drive_base():
    candidates = ['/content/drive/MyDrive', '/content/drive/我的雲端硬碟']
    for c in candidates:
        if os.path.exists(c):
            return c
    if os.path.exists('/content/drive'):
        subs = [os.path.join('/content/drive', d) for d in os.listdir('/content/drive') if not d.startswith('.')]
        if subs:
            return subs[0]
    return '/content/drive/MyDrive'

DRIVE_BASE = get_drive_base()
queue_dir = os.path.join(DRIVE_BASE, 'MediaDownloadQueue')
os.makedirs(queue_dir, exist_ok=True)

# 2. 自動檢查與更新 core_downloader 核心引擎
def ensure_core_downloader():
    try:
        import urllib.request
        raw_url = 'https://raw.githubusercontent.com/hangout1997/Media-Download/main/core_downloader.py'
        print('🔄 正在檢查/同步最新版 core_downloader.py...')
        urllib.request.urlretrieve(raw_url, '/content/core_downloader.py')
        if '/content' not in sys.path:
            sys.path.insert(0, '/content')
        import importlib
        if 'core_downloader' in sys.modules:
            cd_mod = importlib.reload(sys.modules['core_downloader'])
        else:
            import core_downloader as cd_mod
        print('✅ 核心下載引擎 core_downloader 就緒！')
        return cd_mod
    except Exception as e:
        print(f'⚠️ 同步最新模組失敗，嘗試使用現有模組: {e}')
        import core_downloader as cd_mod
        return cd_mod

cd = ensure_core_downloader()

print('=' * 65)
print('🟢 Colab 雲端工作節點已啟動，正在待命中！')
print(f'📂 佇列監聽目錄: {queue_dir}')
print('👉 現在請回到您的 Mac Streamlit 網頁：')
print('   1. 儲存目標選擇『🤖 Google Colab 雲端直存 (0 本地頻寬)』')
print('   2. 貼上網址並點擊開始下載')
print('   3. 本儲存格會自動接單處理，Streamlit 網頁也會同步看到進度條！')
print('=' * 65)

LOCAL_CACHE_DIR = '/content/local_cache'
os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)

def process_one_task(task_path):
    try:
        with open(task_path, 'r', encoding='utf-8') as f:
            task = json.load(f)
    except Exception as e:
        print(f'⚠️ 讀取任務失敗: {e}')
        return

    task_id = task.get('task_id')
    urls = task.get('urls', [])
    save_folder = task.get('save_folder', 'Download')
    only_audio = task.get('only_audio', False)
    cookies = task.get('cookies', {})

    target_dir = os.path.join(DRIVE_BASE, save_folder.strip('/'))
    os.makedirs(target_dir, exist_ok=True)

    status_file = os.path.join(queue_dir, f'status_{task_id}.json')

    def update_status(status, current_item='', pct=0.0, speed_mb=0.0, speed_x=0.0, log='', error=''):
        payload = {
            'task_id': task_id,
            'status': status,
            'current_item': current_item,
            'progress_pct': pct,
            'speed_mb': speed_mb,
            'speed_x': speed_x,
            'last_update': time.time(),
            'log': log,
            'error': error
        }
        try:
            with open(status_file, 'w', encoding='utf-8') as sf:
                json.dump(payload, sf, ensure_ascii=False)
        except Exception:
            pass

    print(f'
📥 [接單] 收到任務 {task_id}，共 {len(urls)} 個網址，直存目標: {target_dir}')
    update_status('running', current_item='正在認領任務...', log='Colab 節點已認領任務')

    success_files = []
    failed_items = []

    for idx, raw_url in enumerate(urls, 1):
        url = cd.normalize_input_url(raw_url)
        print(f'
📍 [{idx}/{len(urls)}] 開始解析: {url}')
        update_status('running', current_item=f'[{idx}/{len(urls)}] 解析網址...', log=f'正在解析 {url}')
        
        try:
            items = cd.get_media_items(url, cookies=cookies)
            if not items:
                err_msg = f'找不到影片串流來源: {url}'
                print(f'❌ {err_msg}')
                failed_items.append((url, err_msg))
                update_status('running', current_item=f'[{idx}/{len(urls)}] 無法解析', log=err_msg)
                continue
                
            for item in items:
                title = item.get('title', 'video')
                clean_title = re.sub(r'[\/\:\*\?"<>\|]', '_', title).strip()
                ext = 'mp3' if only_audio else item.get('ext', 'mp4')
                filename = f'{clean_title}.{ext}'
                
                final_drive_path = os.path.join(target_dir, filename)
                local_temp_path = os.path.join(LOCAL_CACHE_DIR, filename)
                
                if os.path.exists(final_drive_path) and os.path.getsize(final_drive_path) > 0:
                    print(f'⏭️ 雲端已存在，自動跳過: {filename}')
                    success_files.append(filename)
                    update_status('running', current_item=filename, pct=1.0, log=f'已存在於雲端，跳過: {filename}')
                    continue
                    
                print(f'📥 高速下載中: {filename}')
                update_status('running', current_item=filename, pct=0.01, log=f'正在下載 {filename}')
                
                def _prog(pct, cur_dur, tot_dur, speed_mb, speed_x):
                    update_status('running', current_item=filename, pct=pct, speed_mb=speed_mb, speed_x=speed_x,
                                  log=f'{int(pct*100)}% ({speed_mb:.2f} MB/s)')
                    cur_p = int(pct * 100)
                    if cur_p % 10 == 0:
                        print(f'⏳ {filename[:25]}... {cur_p}% | {speed_mb:.2f} MB/s', end='', flush=True)

                if os.path.exists(local_temp_path):
                    try: os.remove(local_temp_path)
                    except: pass

                res_path = cd.download_media_to_path(
                    item, out_path=local_temp_path, force_audio=only_audio, cookies=cookies, progress_cb=_prog
                )
                
                if not os.path.exists(res_path) or os.path.getsize(res_path) == 0:
                    raise RuntimeError(f'檔案下載後大小為 0: {filename}')
                
                print(f'
☁️ 正在複製至 Google Drive: {final_drive_path}...')
                update_status('running', current_item=filename, pct=0.99, log='正在同步至 Google Drive...')
                shutil.copy2(res_path, final_drive_path)
                try: os.remove(res_path)
                except: pass
                
                size_mb = os.path.getsize(final_drive_path) / 1024 / 1024
                print(f'🎉 直存成功: {filename} ({size_mb:.1f} MB)')
                success_files.append(filename)
                update_status('running', current_item=filename, pct=1.0, log=f'直存成功: {filename} ({size_mb:.1f} MB)')

        except Exception as e:
            tb = traceback.format_exc()
            err_line = f'{e}'
            print(f'
❌ 下載失敗: {err_line}
{tb}')
            failed_items.append((url, err_line))
            update_status('running', current_item=f'失敗: {err_line}', log=f'錯誤: {err_line}', error=tb)

    if success_files and not failed_items:
        update_status('completed', current_item=f'已完成 {len(success_files)} 個檔案！', pct=1.0,
                      log=f'成功直存：{', '.join(success_files)}')
        print(f'
✅ 任務 {task_id} 全數成功完成！')
    elif success_files and failed_items:
        update_status('completed', current_item=f'部分完成 ({len(success_files)} 成功, {len(failed_items)} 失敗)', pct=1.0,
                      log=f'成功: {len(success_files)}, 失敗: {len(failed_items)}')
        print(f'
⚠️ 任務 {task_id} 部分成功完成。')
    else:
        all_errs = '; '.join([f'{u}: {msg}' for u, msg in failed_items]) if failed_items else '未找到可用影音串流'
        update_status('failed', current_item='下載任務失敗', pct=0.0,
                      log=f'全數失敗: {all_errs}', error=all_errs)
        print(f'
❌ 任務 {task_id} 處理失敗: {all_errs}')

    try:
        os.remove(task_path)
    except Exception as e:
        print(f'⚠️ 移除 task 檔失敗: {e}')

while True:
    task_files = glob.glob(os.path.join(queue_dir, 'task_*.json'))
    for tf in task_files:
        process_one_task(tf)
    time.sleep(2)



---
## 🖐️ 模式二：直接在 Colab 頁面貼網址下載 (獨立執行)
若您不想打開本地 Streamlit，也可直接在下方表單設定參數並點擊左側三角形執行。

In [ ]:
#@title 🚀 直接在 Colab 執行下載任務
#@markdown ### 網址與儲存設定：
video_urls = "https://www.mvffm.net/movies/your-name/" #@param {type:"string"}
save_folder = "Download" #@param {type:"string"}
only_audio = False #@param {type:"boolean"}

#@markdown ### 選填 Cookie (登入/風控解鎖)：
fb_cookie = "" #@param {type:"string"}
bili_cookie = "" #@param {type:"string"}

import os, time, sys

# 設定 Google Drive 目標儲存路徑
target_dir = os.path.join('/content/drive/MyDrive', save_folder.strip('/'))
os.makedirs(target_dir, exist_ok=True)

urls = [u.strip() for u in video_urls.strip().split('\n') if u.strip()]
cookies = {'fb_cookie': fb_cookie.strip(), 'bili_cookie': bili_cookie.strip()}

print(f"📁 Google Drive 目標資料夾: {target_dir}")
print(f"🔗 待處理網址數量: {len(urls)} 個\n" + "="*60)

for idx, raw_url in enumerate(urls, 1):
    url = cd.normalize_input_url(raw_url)
    print(f"\n📍 [{idx}/{len(urls)}] 正在解析: {url}")
    
    try:
        items = cd.get_media_items(url, cookies=cookies)
        if not items:
            print(f"❌ 找不到可用的串流或影片來源: {url}")
            continue
            
        for item in items:
            title = item.get('title', 'video')
            ext = 'mp3' if only_audio else item.get('ext', 'mp4')
            filename = f"{title}.{ext}"
            out_file = os.path.join(target_dir, filename)
            
            if os.path.exists(out_file):
                print(f"⏭️ 雲端已存在同名檔案，自動跳過: {filename}")
                continue
                
            print(f"📥 開始下載並直存: {filename}")
            
            last_pct = [-1]
            def _progress(pct, cur_dur, tot_dur, speed_mb, speed_x):
                cur_p = int(pct * 100)
                if cur_p != last_pct[0] and cur_p % 5 == 0:
                    last_pct[0] = cur_p
                    print(f"\r⏳ 進度: {cur_p}% | 速度: {speed_mb:.2f} MB/s ({speed_x:.2f}x)", end="", flush=True)
                    
            res_path = cd.download_media_to_path(
                item,
                out_path=out_file,
                force_audio=only_audio,
                cookies=cookies,
                progress_cb=_progress
            )
            print(f"\n🎉 成功直存至 Google Drive: {res_path} ({os.path.getsize(res_path)/1024/1024:.1f} MB)")
            
    except Exception as e:
        print(f"\n❌ 下載失敗 ({url}): {e}")

print("\n" + "="*60 + "\n🎊 所有下載與直存任務處理完畢！可直接至 Google Drive 查看檔案。")